In [2]:
!pip install langchain-community chromadb langchain-groq fastembed unstructured[pdf] python-dotenv
!sudo apt-get update
!sudo apt-get install poppler-utils

  Using cached pydantic-2.13.4-py3-none-any.whl.metadata (109 kB)
  Using cached pydantic_core-2.46.4-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (6.6 kB)
Using cached pydantic-2.13.4-py3-none-any.whl (472 kB)
Using cached pydantic_core-2.46.4-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (2.1 MB)
  Attempting uninstall: pydantic-core
    Found existing installation: pydantic_core 2.41.4
    Uninstalling pydantic_core-2.41.4:
      Successfully uninstalled pydantic_core-2.41.4
  Attempting uninstall: pydantic
    Found existing installation: pydantic 2.12.3
    Uninstalling pydantic-2.12.3:
      Successfully uninstalled pydantic-2.12.3
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 5.50.0 requires aiofiles<25.0,>=22.0, but you have aiofiles 25.1.0 which is incompatible.
gradio 5.50.0 requires pydantic<=2.12.3,>=2.0,

Hit:1 https://cli.github.com/packages stable InRelease
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:5 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:7 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:8 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:9 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:10 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading packag

In [1]:
!pip install pydantic==2.12.3 --force-reinstall
# It is recommended to restart the Colab runtime after running this cell
# to ensure the correct Pydantic version is loaded by all modules.

from langchain_community.document_loaders import UnstructuredPDFLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_community.embeddings.fastembed import FastEmbedEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv

  Using cached pydantic-2.12.3-py3-none-any.whl.metadata (87 kB)
  Using cached annotated_types-0.7.0-py3-none-any.whl.metadata (15 kB)
  Using cached pydantic_core-2.41.4-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (7.3 kB)
  Using cached typing_extensions-4.15.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached typing_inspection-0.4.2-py3-none-any.whl.metadata (2.6 kB)
Using cached pydantic-2.12.3-py3-none-any.whl (462 kB)
Using cached pydantic_core-2.41.4-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (2.1 MB)
Using cached annotated_types-0.7.0-py3-none-any.whl (13 kB)
Using cached typing_extensions-4.15.0-py3-none-any.whl (44 kB)
Using cached typing_inspection-0.4.2-py3-none-any.whl (14 kB)
  Attempting uninstall: typing-extensions
    Found existing installation: typing_extensions 4.15.0
    Uninstalling typing_extensions-4.15.0:
      Successfully uninstalled typing_extensions-4.15.0
  Attempting uninstall: annotated-types
    Found existing in

/tmp/ipykernel_13374/3144089104.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import UnstructuredPDFLoader


In [3]:
load_dotenv()
os.environ["GROQ_API_KEY"]="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"

In [4]:
groq_api_key = os.getenv("GROQ_API_KEY")

In [15]:
from langchain_community.document_loaders.onedrive_file import CHUNK_SIZE
#Load the documents
def load_documents(file_path):
  loader = UnstructuredPDFLoader(file_path)
  documents = loader.load()
  return documents

def split_documents(documents):
  text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap = 100)
  docs = text_splitter.split_documents(documents)
  return docs

def initialize_embeddings(model_name= "BAAI/bge-base-en-v1.5"):
  embed_model = FastEmbedEmbeddings(model_name=model_name)
  return embed_model

def create_vectorstore(documents, embed_model, persist_directory="chroma_db", collection_name="my_collection"):
  vectorstore = Chroma.from_documents(
      documents=documents,
      embedding=embed_model,
      persist_directory=persist_directory,
      collection_name=collection_name
      )
  return vectorstore

def initialize_chat_model(temperature=0, model_name="openai/gpt-oss-20b", api_key=None):
    chat_model = ChatGroq(
        temperature=temperature,
        model_name=model_name,
        api_key=api_key
        )
    return chat_model

In [16]:
documents = load_documents ("/content/HIS100E World History I.pdf")

In [17]:
chunked_document = split_documents(documents)
print("length of chunked document is: ", len(chunked_document))

length of chunked document is:  36


In [18]:
embed_model = initialize_embeddings()
vectorstore = create_vectorstore(chunked_document, embed_model)
retriever = vectorstore.as_retriever (search_kwargs={"k":36})

In [19]:
from langchain_core.prompts import PromptTemplate
from langchain_classic.chains import RetrievalQA


In [20]:
class QAretrieval:
  def __init__(self, vectorstore,chat_model,search_kwargs=None):
    if search_kwargs is None:
      search_kwargs = {"k":3}
    self.retriever = vectorstore.as_retriever(search_kwargs=search_kwargs)
    self.chat_model = chat_model
    self.custom_prompt_template ="""
      Use the following pieces of context to answer the user's question. if you don't know the answer, just say that you don't know, don't try to make up an answer.

      Context:{context}
      Question: {question}

      Only return the helpful answer below and nothing else.
      Helpful answer:
      """
    self.prompt= self.set_custom_prompt()

  def set_custom_prompt(self):
    prompt = PromptTemplate(template=self.custom_prompt_template, input_variables=["context","question"])
    return prompt

  def perform_qa(self,query):
    qa = RetrievalQA.from_chain_type(llm=self.chat_model,
                                           chain_type="stuff",
                                           retriever=self.retriever,
                                           return_source_documents=False,
                                           chain_type_kwargs={"prompt":self.prompt})
    response = qa.invoke({"query":query})
    return response

In [26]:
chat_model = initialize_chat_model(api_key=groq_api_key)
qa_retrieval = QAretrieval(vectorstore, chat_model)

In [ ]:
while True:
  user_query = input ("please enter your questions")

  if user_query.lower() =="exit":
    print("exiting ....")
    break

  try:
    response =qa_retrieval.perform_qa(user_query)
    print("Response", response)
  except Exception as e:
    print(f"An error occured: {e}")

please enter your questionswhat is ancient india
Response {'query': 'what is ancient india', 'result': '**Ancient India** refers to the early civilizations and cultural developments that emerged in the Indian subcontinent from roughly the 3rd millennium\u202fBCE to the early centuries\u202fCE. It is most famously marked by the **Indus Valley Civilization** (c.\u202f2600–1900\u202fBCE), one of the world’s earliest urban societies, followed by the **Vedic period** (c.\u202f1500–500\u202fBCE) when the sacred texts of the Vedas were composed and the foundations of Hindu social and religious structures were laid. These eras collectively illustrate India’s long and complex history, with scholars often dating the origins of its civilization to around 5,000\u202fBCE, aligning with the broader timeline of human civilization in the region.'}
please enter your questionsancient greece
Response {'query': 'ancient greece', 'result': 'Ancient Greece (c.\u202f800\u202f–\u202f146\u202fBCE) was a collec